# NeuroForecast — reproduce the result, including the failure

**Day-7 trust triage for neuronal network assays.** AI4S Open Innovation: AI for Life Science, category **Tool & Platform**.

A researcher dosing neuronal cultures on microelectrode arrays measures activity on day 7, days before the network finishes developing. This tool answers, on day 7:

1. Can I trust today's measurement?
2. Can this condition's day-12 outcome be forecast at all?
3. Where should a limited review budget go?

**The headline, stated first.** We sealed three experiment dates (224 conditions, 32 chemicals absent from training), wrote the success criteria into a hashed protocol, and scored them **once**. The learned forecast **lost** to carrying the day-7 readout forward: date-macro MAE **0.7716 versus 0.6088**. We claim no forecasting advantage over simple methods.

What *did* generalize to those unseen batches and chemicals is the reliability layer, and this notebook reproduces all of it from the evidence tables in the repository — no 160 MB download needed for parts 1 to 5.

Repository: <https://github.com/SaltyTaro/neuroforecast>

> On Kaggle, enable **Internet** in the notebook settings so the first cell can clone the repository.

## 0. Setup

Clones the public repository if it is not already present and puts its `tools/` on the import path. Nothing here needs a GPU, an account, or a paid service.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/SaltyTaro/neuroforecast.git"
root = Path("neuroforecast")
if Path("tools/neuroforecast_triage.py").exists():
    root = Path(".")                      # already inside a clone
elif not root.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)

root = root.resolve()
sys.path.insert(0, str(root / "tools"))
print("repository:", root)
print("files:", sum(1 for _ in root.rglob("*") if _.is_file()))

## 1. Do the published numbers match the evidence?

This is the check to run first. `verify_published_numbers.py` recomputes every headline number in the README and the reports **directly from the per-case tables** in `evaluation/`, never from a summary file, and compares each at the precision it is quoted to. If any claim in the write-up has drifted from the evidence, this fails.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-X", "utf8", "-B", str(root / "tools/verify_published_numbers.py")],
                        capture_output=True, text=True, cwd=root)
print(result.stdout[-2000:] or result.stderr[-2000:])
assert result.returncode == 0, 'a published number does not match the evidence'

## 2. The tool on a batch that passes quality control

`examples/reserve_20171011_usable_day5_day7.csv` holds real day-7 wells from one of the sealed reserve dates. The shipped models never saw this batch or any of its chemicals.

The entry point **rejects** day-9 and day-12 rows, so a forecast cannot be produced with information the researcher would not have on day 7.

In [ ]:
import pandas as pd
import neuroforecast_triage as triage

bundle = triage.load_bundle(root / "models", root / "models/development_lock.json", root / "models/reserve_summary.json")
observed = pd.read_csv(root / "examples/reserve_20171011_usable_day5_day7.csv", dtype={"identity": str})
print("input rows:", len(observed), "| days present:", sorted(observed.DIV.unique()))

table, summary = triage.triage(observed, bundle)
batch = summary["batches"][0]
print("\nbatch", batch["batch"], "-", batch["batch_verdict"])
print("day-7 reference activity:", batch["day7_reference_activity_network_spikes"], "network spikes")
print("conditions:", batch["conditions"], "| declined:", batch["forecasts_declined"], "| quiet at day 7:", batch["quiet_at_day7"])

Two ranked lists come out of this. The one that matters is **early warning**: conditions that have barely moved by day 7, where the early readout carries no information. The other list is dominated by conditions that have already collapsed — which the day-7 column shows directly, without a model.

In [ ]:
columns = ["name", "dose", "day7_observed_log2", "forecast_log2", "forecast_low", "forecast_high", "persistence_log2", "verdict"]
early = table.loc[table.early_warning_flagged].sort_values("early_warning_rank")
print("EARLY WARNING - quiet at day 7, forecast to change")
display(early[columns].round(2).reset_index(drop=True))

print("\nLARGEST FORECAST CHANGE OVERALL - mostly conditions that already collapsed by day 7")
display(table.loc[table.review_flagged].sort_values("review_rank")[columns].round(2).head(6).reset_index(drop=True))

## 3. The batch where the forecast failed, and the flag that caught it

On experiment date 20171004 every plate had a day-7 control median of **zero** network spikes: the cultures had not begun firing when the input measurement was taken. They recovered to medians of 23–54 by day 12.

This is the batch where the model produced its worst errors. The quality flag, with a threshold fixed on development data before the reserve was opened, fires on all 84 of its conditions — using only day-7 information.

In [ ]:
bad = pd.read_csv(root / "examples/reserve_20171004_low_quality_day5_day7.csv", dtype={"identity": str})
bad_table, bad_summary = triage.triage(bad, bundle)
batch = bad_summary["batches"][0]

print("batch", batch["batch"])
print("day-7 reference activity:", batch["day7_reference_activity_network_spikes"], "network spikes")
print("VERDICT:", batch["batch_verdict"])
print("conditions carrying a quality flag:", batch["conditions_with_any_quality_flag"], "of", batch["conditions"])
print("low-reference-activity fraction:", batch["low_reference_activity_fraction"])

Every condition the tool surfaces here is marked **declined**. The tool's answer for this batch is "do not trust a forecast; measure day 12 directly" — which is the correct call, reached before any day-12 data existed.

In [ ]:
surfaced = bad_table.loc[bad_table.early_warning_flagged].sort_values("early_warning_rank")
display(surfaced[columns].round(2).head(8).reset_index(drop=True))
print("surfaced conditions:", len(surfaced), "| of which declined:", int((~surfaced.forecast_trusted).sum()))

## 4. The headline comparison, recomputed per date

Date-macro MAE: absolute error averaged over doses within a chemical, then over chemicals within a date, then equally over dates. Lower is better.

In [ ]:
import neuroforecast_gate as gate

predictions = pd.read_csv(root / "evaluation/reserve_predictions.csv", dtype={"identity": str})
labels = {
    "persistence_day7": "Carry day 7 forward (simple rule)",
    "extra_trees_relative_day7": "Learned forecast (this project)",
    "extra_trees_activity_coordination": "Smaller activity/coordination model",
    "control_reference": "Assume control-like day 12",
}
rows = {}
for model, label in labels.items():
    losses = gate.date_losses(predictions.loc[predictions.model == model])
    rows[label] = {**{str(date): round(value, 3) for date, value in losses.items()}, "mean of dates": round(float(losses.mean()), 4)}

comparison = pd.DataFrame(rows).T
comparison.index.name = "method"
display(comparison)
print("\nthe simple rule is more accurate overall:",
      comparison.loc["Carry day 7 forward (simple rule)", "mean of dates"] < comparison.loc["Learned forecast (this project)", "mean of dates"])

Persistence wins overall and on two of the three dates. The learned model beats the smaller activity/coordination model on all three, which is the less interesting comparison. The reserved batches are simply more persistent than development: the day-7/day-12 correlation is 0.87 there against 0.54 on development.

## 5. What generalized: the trust verdict

A difficulty model reading only day-7 inputs estimates whether a condition's outcome is predictable at all. With the threshold locked on development data, it declines 14% of reserve conditions — and those carry about half of all forecast error.

It improves the simple rule too, so the verdict is useful whichever predictor is used.

In [ ]:
import json

intervals = pd.read_csv(root / "evaluation/reserve_intervals.csv", dtype={"identity": str})
threshold = json.loads((root / "models/development_lock.json").read_text(encoding="utf-8"))["locked_thresholds"]["abstention_sigma_p80"]
retained = intervals.sigma <= threshold

learned = abs(intervals.prediction - intervals.target)
simple = predictions.loc[predictions.model == "persistence_day7"].set_index("case_id").reindex(intervals.case_id)
simple = abs(simple.prediction.to_numpy() - simple.target.to_numpy())

summary = pd.DataFrame({
    "learned forecast": [learned.mean(), learned[retained].mean(), learned[~retained].mean()],
    "simple rule": [simple.mean(), simple[retained.to_numpy()].mean(), simple[~retained.to_numpy()].mean()],
}, index=["all 224 conditions", f"kept ({int(retained.sum())})", f"declined ({int((~retained).sum())})"]).round(3)
display(summary)

print("share of total forecast error inside the declined conditions: %.0f%%" % (100 * learned[~retained].sum() / learned.sum()))
inside = (intervals.target >= intervals["scaled_lower_0.8"]) & (intervals.target <= intervals["scaled_upper_0.8"])
print("interval coverage observed at a nominal 80%% level: %.1f%%" % (100 * inside.mean()))

## 6. Optional: full reproduction from the public archive

Parts 1–5 used the audited per-case tables. This part rebuilds them from the raw EPA archive (~160 MB, public domain). It takes roughly 10 minutes: `develop` is single-threaded and bit-reproducible, `reserve` takes 12 seconds and refuses to run twice without the unchanged development lock.

Set `RUN_FULL = True` to execute it.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    import subprocess, sys
    steps = [
        ["tools/fetch_epa_data.py", "--out", "raw/epa_nfa_raw.zip"],
        ["tools/neuroforecast_gate.py", "--stage", "prepare", "--raw", "raw/epa_nfa_raw.zip", "--out", "runs/gate"],
        ["tools/neuroforecast_gate.py", "--stage", "develop", "--out", "runs/gate"],
        ["tools/neuroforecast_gate.py", "--stage", "reserve", "--unseal", "--out", "runs/gate"],
        ["tools/neuroforecast_gate.py", "--stage", "audit", "--raw", "raw/epa_nfa_raw.zip", "--out", "runs/gate"],
    ]
    for step in steps:
        print("\n$", " ".join(step))
        result = subprocess.run([sys.executable, "-X", "utf8", "-B", *step], cwd=root, capture_output=True, text=True)
        print(result.stdout[-1500:] or result.stderr[-1500:])
        assert result.returncode == 0, step
else:
    print("set RUN_FULL = True to rebuild every table from the public archive (about 10 minutes)")

## What this does not establish

Rat cortical cultures in multi-well plates are not perfused organ chips, human cells, or clinical toxicity. Nothing here validates saved experimental days or wells, irreversibility, recovery after washout, or chip performance. The reserve is three later batches from one public study, not broad external validity, and the warning evidence rests on ten positive cases in a single batch. Forecast magnitudes are not calibrated probabilities. Prediction intervals observed 71.4% coverage at a nominal 80% level — a real shortfall under batch shift.

Early neural forecasting, MEA chemical screening and toxicological tipping-point analysis are established prior work; Extra Trees, control normalization and split-conformal prediction are standard methods used as implementation choices. The contribution is the triage workflow and the evaluation protocol around it.

Data: EPA network formation assay, [DOI 10.23719/1503191](https://doi.org/10.23719/1503191), accompanying [Shafer et al., 2019](https://doi.org/10.1093/toxsci/kfz052); U.S. public domain under the EPA ScienceHub licence statement. Code MIT.